# Driver-Independent Suppression of Nuclear-Encoded Mitochondrial Metabolism at Stage I in Clear Cell Renal Cell Carcinoma: A Multi-Cohort Transcriptomic Analysis

**Authors:** Aroob A. Gomosani, Haneen A. Marghalani, Layal M. Al Matar  
**Institution:** King Abdulaziz University, Jeddah, Saudi Arabia  
**Correspondence:** agomosani0004@stu.kau.edu.sa

---

## Overview

This notebook contains the complete analysis pipeline for the KIRC mitochondrial transcriptomics study. It reproduces all results reported in the manuscript.

**Pipeline summary:**
1. TCGA-KIRC discovery — MitoCarta 3.0-restricted differential expression
2. Stage I analysis — earliest-stage dysregulation
3. Driver independence — VHL and PBRM1 stratification
4. 70-gene core signature — four-way intersection
5. Candidate scoring and locking (pre-specified, before any clinical testing)
6. Clinical, survival, and ROC analyses
7. GSEA pathway analysis
8. External validation — GSE53757 and GSE40435
9. DESeq2 methodological cross-validation
10. Pan-cohort convergence analysis

**Data sources (all publicly available):**
- TCGA-KIRC: [UCSC Xena](https://xenabrowser.net) — FPKM-UQ, STAR counts, somatic mutations, clinical, survival
- MitoCarta 3.0: [Broad Institute](https://www.broadinstitute.org/mitocarta)
- GSE53757: [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE53757)
- GSE40435: [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE40435)

**Environment:** Python 3.12 · Google Colab (recommended) or local Jupyter


## 1. Environment Setup

In [ ]:
# Install dependencies (run once)
!pip install lifelines gseapy pydeseq2 GEOparse adjustText -q


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
print("Environment ready.")


## 2. Configuration

> **⚙️ Edit this cell before running.** Set `DATA_FOLDER` to the directory containing your downloaded data files. All other paths are derived from this variable.
>
> **Expected files in DATA_FOLDER:**
> - `Human.MitoCarta3.0.xls`
> - `TCGA-KIRC.star_fpkm-uq.tsv.gz`
> - `TCGA-KIRC.star_counts.tsv.gz`
> - `TCGA_KIRC_Clinical.tsv`
> - `KIRC_master_clinical.csv`
> - `TCGA-KIRC.somaticmutation_wxs.tsv.gz`
> - `TCGA-KIRC.survival.tsv.gz`
> - `GSE53757_series_matrix.txt.gz`
> - `GSE40435_series_matrix.txt.gz`


In [ ]:
# ── USER CONFIGURATION — edit DATA_FOLDER to match your setup ──────────────
# Google Colab example:
#   DATA_FOLDER = '/content/drive/MyDrive/kirc_data/'
# Local example:
#   DATA_FOLDER = '/home/user/kirc_data/'

DATA_FOLDER = '/content/drive/MyDrive/kirc_data/'   # <-- CHANGE THIS

# Mount Google Drive (Colab only — remove if running locally)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── File paths (derived automatically from DATA_FOLDER) ────────────────────
MITOCARTA_FILE    = DATA_FOLDER + 'Human.MitoCarta3.0.xls'
EXPR_FPKM_UQ_FILE = DATA_FOLDER + 'TCGA-KIRC.star_fpkm-uq.tsv.gz'
EXPR_COUNTS_FILE  = DATA_FOLDER + 'TCGA-KIRC.star_counts.tsv.gz'
CLINICAL_FILE     = DATA_FOLDER + 'TCGA_KIRC_Clinical.tsv'
MASTER_CLINICAL_FILE = DATA_FOLDER + 'KIRC_master_clinical.csv'
MUTATION_FILE     = DATA_FOLDER + 'TCGA-KIRC.somaticmutation_wxs.tsv.gz'
SURVIVAL_FILE     = DATA_FOLDER + 'TCGA-KIRC.survival.tsv.gz'
GSE53757_FILE     = DATA_FOLDER + 'GSE53757_series_matrix.txt.gz'
GSE40435_FILE     = DATA_FOLDER + 'GSE40435_series_matrix.txt.gz'
OUT_PREFIX        = DATA_FOLDER  # output files saved here

# ── Analysis thresholds (pre-specified, do not change) ──────────────────────
PADJ_THRESHOLD = 0.05
LOG2FC_STRICT  = 1.0
LOG2FC_LENIENT = 0.5
DRIVERS        = ['VHL', 'PBRM1', 'SETD2', 'BAP1']
RANDOM_SEED    = 42

# ── File existence check ────────────────────────────────────────────────────
files_to_check = [
    ('MitoCarta',        MITOCARTA_FILE),
    ('FPKM-UQ',          EXPR_FPKM_UQ_FILE),
    ('STAR counts',      EXPR_COUNTS_FILE),
    ('Clinical',         CLINICAL_FILE),
    ('Master clinical',  MASTER_CLINICAL_FILE),
    ('Mutations',        MUTATION_FILE),
    ('Survival',         SURVIVAL_FILE),
    ('GSE53757',         GSE53757_FILE),
    ('GSE40435',         GSE40435_FILE),
]
all_ok = True
for label, path in files_to_check:
    if os.path.exists(path):
        print(f"  ✓  {label:18s}  {os.path.basename(path)}")
    else:
        all_ok = False
        print(f"  ✗  {label:18s}  MISSING — {path}")
if all_ok:
    print("\nAll files found. Ready to run.")
else:
    print("\n⚠  Fix missing files before continuing.")


## 3. Data Loading

In [ ]:
# Load FPKM-UQ expression matrix
expression_fpkm = pd.read_csv(EXPR_FPKM_UQ_FILE, sep='\t', compression='gzip')
print(f"Expression matrix: {expression_fpkm.shape}")
print(f"Value range: {expression_fpkm.iloc[:,1:].min().min():.2f} – "
      f"{expression_fpkm.iloc[:,1:].max().max():.2f}  (log2 scale expected ~0–22)")


In [ ]:
# MitoCarta 3.0 filter — restrict to nuclear-encoded mitochondrial genes
mitocarta = pd.read_excel(MITOCARTA_FILE, sheet_name='A Human MitoCarta3.0', engine='xlrd')
print(f"MitoCarta entries: {len(mitocarta)}")  # expected: 1136

expression_fpkm['Ensembl_ID_clean'] = expression_fpkm['Ensembl_ID'].str.split('.').str[0]
mito_ids = mitocarta['EnsemblGeneID_mapping_version_20200130'].dropna().astype(str).tolist()

fpkm_mito = expression_fpkm[expression_fpkm['Ensembl_ID_clean'].isin(mito_ids)].copy()
fpkm_mito = fpkm_mito.drop(columns=['Ensembl_ID']).rename(columns={'Ensembl_ID_clean': 'Ensembl_ID'})
fpkm_mito = fpkm_mito.reset_index(drop=True)

zero_mask = (fpkm_mito.drop(columns=['Ensembl_ID']) == 0).all(axis=1)
print(f"All-zero genes removed: {zero_mask.sum()}")
fpkm_mito = fpkm_mito[~zero_mask].reset_index(drop=True)
print(f"MitoCarta genes retained: {len(fpkm_mito)}  "
      f"({100*len(fpkm_mito)/len(mito_ids):.1f}% match rate)")
# Expected: 1074 genes (94.5% match rate)


In [ ]:
# Load clinical, mutation, and survival data
master = pd.read_csv(MASTER_CLINICAL_FILE)
mutation_long = pd.read_csv(MUTATION_FILE, sep='\t', compression='gzip')
survival = pd.read_csv(SURVIVAL_FILE, sep='\t', compression='gzip')
print(f"Master clinical: {master.shape}")
print(f"Mutation events:  {mutation_long.shape}")
print(f"Survival records: {survival.shape}")

# Build binary driver mutation matrix
gene_col   = 'gene'   if 'gene'   in mutation_long.columns else 'Hugo_Symbol'
sample_col = 'sample' if 'sample' in mutation_long.columns else 'Tumor_Sample_Barcode'
driver_status = pd.DataFrame({'sample': mutation_long[sample_col].unique()})
for drv in DRIVERS:
    affected = mutation_long.loc[mutation_long[gene_col] == drv, sample_col].unique()
    driver_status[f'{drv}_mut'] = driver_status['sample'].isin(affected).astype(int)
print(f"\nDriver mutation counts (all stages):")
print(driver_status[[f'{d}_mut' for d in DRIVERS]].sum())


## 4. Sample Classification & Independence Check

In [ ]:
all_cols = list(fpkm_mito.columns)
tumor_samples  = [c for c in all_cols if c.endswith('-01A')]
normal_samples = [c for c in all_cols if c.endswith('-11A')]

def patient_id(barcode): return barcode[:12]

tumor_patients   = {patient_id(s) for s in tumor_samples}
normal_patients  = {patient_id(s) for s in normal_samples}
overlap_patients = tumor_patients & normal_patients

print(f"Tumour (-01A) samples:  {len(tumor_samples)}")   # expected: 529
print(f"Normal (-11A) samples:  {len(normal_samples)}")   # expected: 72
print(f"Overlapping patients:   {len(overlap_patients)}")  # expected: 71

# Independent tumour set — excludes patients also contributing normals
tumor_samples_indep = [s for s in tumor_samples if patient_id(s) not in overlap_patients]
print(f"Independent tumour set: {len(tumor_samples_indep)}")  # expected: 458

# Gene symbol map for downstream use
gene_names = mitocarta[['EnsemblGeneID_mapping_version_20200130', 'Symbol']].copy()
gene_names.columns = ['Ensembl_ID', 'gene_symbol']


## 5. Quality Control — PCA

PCA is performed **before** any differential expression analysis to confirm
tumour-normal separation and absence of batch effects.

In [ ]:
sample_cols = tumor_samples + normal_samples
X = fpkm_mito[sample_cols].T.values
labels = np.array(['Tumour']*len(tumor_samples) + ['Normal']*len(normal_samples))

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_SEED)
pcs = pca.fit_transform(X_scaled)
var_explained = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(7, 6))
for grp, col in [('Normal', '#2E86AB'), ('Tumour', '#C73E1D')]:
    mask = labels == grp
    ax.scatter(pcs[mask,0], pcs[mask,1], c=col, alpha=0.6, s=18,
               label=f"{grp} (n={mask.sum()})")
ax.set_xlabel(f"PC1 — {var_explained[0]:.1f}% variance")
ax.set_ylabel(f"PC2 — {var_explained[1]:.1f}% variance")
ax.set_title("PCA QC — TCGA-KIRC MitoCarta-restricted expression")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_PREFIX + 'KIRC_PCA_QC.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"PC1: {var_explained[0]:.1f}% variance  (expected ~21%)")


## 6. All-Stage Differential Expression

Mann-Whitney U test + Benjamini-Hochberg FDR across all 1,074 MitoCarta genes.  
Dual threshold: **adjusted p < 0.05 AND |log2FC| > 1.0** (strict) or **> 0.5** (lenient).


In [ ]:
tumor_vals  = fpkm_mito[tumor_samples].values
normal_vals = fpkm_mito[normal_samples].values

results = []
for i, ens in enumerate(fpkm_mito['Ensembl_ID']):
    t = tumor_vals[i]; n = normal_vals[i]
    stat, p = stats.mannwhitneyu(n, t, alternative='two-sided')
    results.append({'Ensembl_ID': ens,
                    'mean_normal': np.mean(n), 'mean_tumor': np.mean(t),
                    'log2FC': np.mean(t) - np.mean(n), 'pvalue': p})

de_all = pd.DataFrame(results)
_, padj, _, _ = multipletests(de_all['pvalue'], method='fdr_bh')
de_all['pval_adj']   = padj
de_all['sig_strict'] = (de_all['pval_adj'] < PADJ_THRESHOLD) & (de_all['log2FC'].abs() > LOG2FC_STRICT)
de_all['sig_lenient']= (de_all['pval_adj'] < PADJ_THRESHOLD) & (de_all['log2FC'].abs() > LOG2FC_LENIENT)
de_all['rank_score'] = -np.log10(de_all['pval_adj'].replace(0, 1e-300)) * np.sign(de_all['log2FC'])
de_all = de_all.merge(gene_names, on='Ensembl_ID', how='left')

n_strict = de_all['sig_strict'].sum()
n_down = ((de_all['sig_strict']) & (de_all['log2FC'] < 0)).sum()
n_up   = ((de_all['sig_strict']) & (de_all['log2FC'] > 0)).sum()
print(f"Strict  (|log2FC|>{LOG2FC_STRICT}): {n_strict}  (↓{n_down}  ↑{n_up})")
# Expected: 111 (93↓, 18↑)
print(f"Lenient (|log2FC|>{LOG2FC_LENIENT}): {de_all['sig_lenient'].sum()}")
# Expected: 369 (inclusive of strict)
de_all.to_csv(OUT_PREFIX + 'KIRC_DE_all_stage.csv', index=False)


In [ ]:
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(9, 7))
de_all['nlog10_padj'] = -np.log10(de_all['pval_adj'].replace(0, 1e-300))
colors_arr = np.where(de_all['sig_strict'] & (de_all['log2FC'] > 0), '#C73E1D',
              np.where(de_all['sig_strict'] & (de_all['log2FC'] < 0), '#2E86AB',
              np.where(de_all['sig_lenient'], '#BBBBBB', '#DDDDDD')))
ax.scatter(de_all['log2FC'], de_all['nlog10_padj'], c=colors_arr, alpha=0.6, s=15)
ax.axhline(-np.log10(PADJ_THRESHOLD), ls='--', color='black', lw=0.6)
for x in [-LOG2FC_STRICT, LOG2FC_STRICT]:
    ax.axvline(x, ls='--', color='black', lw=0.6)
top10_v = de_all[de_all['sig_strict']].nlargest(10, 'log2FC')
bot10_v = de_all[de_all['sig_strict']].nsmallest(10, 'log2FC')
labs = [ax.text(r['log2FC'], r['nlog10_padj'], r['gene_symbol'], fontsize=8)
        for _, r in pd.concat([top10_v, bot10_v]).iterrows()]
adjust_text(labs, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))
ax.set_xlabel("log2 Fold Change (Tumour vs Normal)")
ax.set_ylabel("−log10 Adjusted p-value")
ax.set_title("KIRC MitoCarta Differential Expression — All-Stage")
plt.tight_layout()
plt.savefig(OUT_PREFIX + 'KIRC_volcano_all_stage.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Stage I Analysis

Tests whether mitochondrial dysregulation is present at the **earliest diagnosable
disease stage**. The 71 patients contributing both tumour and normal samples are
excluded to ensure statistical independence (yielding n = 246 independent Stage I tumours).


In [ ]:
stage1_mask = (
    master['ajcc_pathologic_stage.diagnoses'].isin(['Stage I', 'Stage IA', 'Stage IB']) &
    (master['sample_type.samples'] == 'Primary Tumor')
)
stage1_ids    = master.loc[stage1_mask, 'sample'].tolist()
stage1_in_expr = [s for s in stage1_ids if s in fpkm_mito.columns]
stage1_indep   = [s for s in stage1_in_expr if patient_id(s) not in overlap_patients]
print(f"Stage I total:       {len(stage1_in_expr)}")  # expected: 270
print(f"Stage I independent: {len(stage1_indep)}")    # expected: 246

def run_de(group_a_cols, group_b_cols, label_a='A', label_b='B'):
    rows = []
    A = fpkm_mito[group_a_cols].values
    B = fpkm_mito[group_b_cols].values
    for i, ens in enumerate(fpkm_mito['Ensembl_ID']):
        _, p = stats.mannwhitneyu(A[i], B[i], alternative='two-sided')
        rows.append({'Ensembl_ID': ens,
                     'log2FC': np.mean(B[i]) - np.mean(A[i]), 'pvalue': p})
    df = pd.DataFrame(rows)
    _, padj, _, _ = multipletests(df['pvalue'], method='fdr_bh')
    df['pval_adj']    = padj
    df['sig_strict']  = (df['pval_adj'] < PADJ_THRESHOLD) & (df['log2FC'].abs() > LOG2FC_STRICT)
    df['sig_lenient'] = (df['pval_adj'] < PADJ_THRESHOLD) & (df['log2FC'].abs() > LOG2FC_LENIENT)
    df['rank_score']  = -np.log10(df['pval_adj'].replace(0, 1e-300)) * np.sign(df['log2FC'])
    return df.merge(gene_names, on='Ensembl_ID', how='left')

# Primary (independent) and sensitivity (full) analyses
de_stage1_indep = run_de(normal_samples, stage1_indep, 'normal', 'stage1')
de_stage1_full  = run_de(normal_samples, stage1_in_expr, 'normal', 'stage1')

# Overlap between Stage I and all-stage
sig_stage1   = set(de_stage1_indep.loc[de_stage1_indep['sig_strict'], 'Ensembl_ID'])
sig_all      = set(de_all.loc[de_all['sig_strict'], 'Ensembl_ID'])
overlap_s1   = len(sig_stage1 & sig_all)
s1_specific  = sig_stage1 - sig_all
s1_spec_syms = de_stage1_indep.loc[
    de_stage1_indep['Ensembl_ID'].isin(s1_specific), 'gene_symbol'].tolist()

print(f"\nStage I strict: {de_stage1_indep['sig_strict'].sum()}")  # expected: 101
print(f"Overlap with all-stage: {overlap_s1}/101 (97%)")            # expected: 98
print(f"Stage I-specific genes (not in all-stage): {s1_spec_syms}")
# Expected: PNKD, CYCS, ACSM5
sensitivity_overlap = len(set(de_stage1_indep.loc[de_stage1_indep['sig_strict'],'Ensembl_ID']) &
                           set(de_stage1_full.loc[de_stage1_full['sig_strict'],'Ensembl_ID']))
print(f"Sensitivity check: {sensitivity_overlap}/101 retained with all Stage I (100%)")
de_stage1_indep.to_csv(OUT_PREFIX + 'KIRC_DE_stage1_independent.csv', index=False)


## 8. Driver Mutation Independence

For each major driver (VHL, PBRM1), Stage I tumours are stratified into
mutation-positive and mutation-intact subgroups. Genes significant in **both**
subgroups are classified as driver-independent.

Minimum n = 10 mutated samples required for analysis; SETD2 (n = 14) meets this
threshold but is treated as exploratory given limited power. BAP1 (n = 5) is excluded.


In [ ]:
master_d = master.merge(driver_status, on='sample', how='left')
for d in DRIVERS:
    master_d[f'{d}_mut'] = master_d[f'{d}_mut'].fillna(-1).astype(int)

driver_results = {}
for d in DRIVERS:
    s1_mut    = master_d.loc[(master_d['sample'].isin(stage1_indep)) &
                              (master_d[f'{d}_mut'] == 1), 'sample'].tolist()
    s1_intact = master_d.loc[(master_d['sample'].isin(stage1_indep)) &
                              (master_d[f'{d}_mut'] == 0), 'sample'].tolist()
    s1_mut    = [s for s in s1_mut    if s in fpkm_mito.columns]
    s1_intact = [s for s in s1_intact if s in fpkm_mito.columns]
    print(f"\n{d}: n_mut={len(s1_mut)}, n_intact={len(s1_intact)}")
    if len(s1_mut) < 10:
        print(f"  Skipping (< 10 mutated samples)")
        continue
    de_mut    = run_de(normal_samples, s1_mut,    'normal', f'{d}_mut')
    de_intact = run_de(normal_samples, s1_intact, 'normal', f'{d}_intact')
    sig_mut    = set(de_mut.loc[de_mut['sig_strict'], 'Ensembl_ID'])
    sig_intact = set(de_intact.loc[de_intact['sig_strict'], 'Ensembl_ID'])
    indep = sig_mut & sig_intact
    print(f"  Independent: {len(indep)}  |  {d}-specific: {len(sig_mut - sig_intact)}")
    driver_results[d] = {
        'mut': de_mut, 'intact': de_intact,
        'independent': indep, 'specific': sig_mut - sig_intact,
        'n_mut': len(s1_mut), 'n_intact': len(s1_intact)
    }
# Expected: VHL n_mut=84 → 81 independent; PBRM1 n_mut=71 → 75 independent
# SETD2 n_mut=14 (exploratory); BAP1 n_mut=5 (skipped)


## 9. Core Signature Construction

**Four-way intersection** (pre-specified before any clinical or validation analysis):
1. All-stage strict-significant
2. Stage I independent strict-significant
3. VHL-independent
4. PBRM1-independent

This yields the **70-gene dual driver-independent core signature**.


In [ ]:
sig_vhl_indep  = driver_results['VHL']['independent']
sig_pbrm1_indep = driver_results['PBRM1']['independent']
sig_setd2_indep = driver_results.get('SETD2', {}).get('independent', set())

# Primary core (VHL + PBRM1)
core_signature = sig_all & sig_stage1 & sig_vhl_indep & sig_pbrm1_indep
# Sensitivity (adds SETD2 — exploratory)
core_signature_setd2 = sig_all & sig_stage1 & sig_vhl_indep & sig_pbrm1_indep & sig_setd2_indep

print(f"All-stage strict:                {len(sig_all)}")
print(f"Stage I strict (independent):    {len(sig_stage1)}")
print(f"VHL-independent:                 {len(sig_vhl_indep)}")
print(f"PBRM1-independent:               {len(sig_pbrm1_indep)}")
print(f"Core (VHL + PBRM1):              {len(core_signature)}")   # expected: 70
print(f"Sensitivity (+SETD2):            {len(core_signature_setd2)}")  # expected: 62

core_df = de_all[de_all['Ensembl_ID'].isin(core_signature)].sort_values('log2FC')
core_df.to_csv(OUT_PREFIX + 'KIRC_core_signature_70genes.csv', index=False)

n_down_core = (core_df['log2FC'] < 0).sum()
n_up_core   = (core_df['log2FC'] > 0).sum()
print(f"\nCore: {n_down_core} downregulated, {n_up_core} upregulated")
# Expected: 68↓, 2↑ (COX4I2 and FKBP10 upregulated)


## 10. Candidate Gene Scoring and Locking

Pre-specified composite scoring formula applied to the 70-gene core:

```
score = 0.40 × norm(|log2FC all-stage|) + 0.40 × norm(|log2FC Stage I|) + 0.20 × (1 − norm(cross-driver SD))
```

The **top 10 genes are locked here** — no further changes after this cell.
All subsequent clinical, survival, ROC, and validation analyses use these
10 candidates unchanged.


In [ ]:
candidates = de_all[de_all['Ensembl_ID'].isin(core_signature)].copy()
candidates = candidates.merge(
    de_stage1_indep[['Ensembl_ID','log2FC']].rename(columns={'log2FC':'log2FC_stage1'}),
    on='Ensembl_ID', how='left')
for d in DRIVERS:
    if d in driver_results:
        candidates = candidates.merge(
            driver_results[d]['mut'][['Ensembl_ID','log2FC']].rename(
                columns={'log2FC':f'log2FC_{d}'}), on='Ensembl_ID', how='left')

driver_fc_cols = [f'log2FC_{d}' for d in DRIVERS if f'log2FC_{d}' in candidates.columns]
candidates['cross_driver_sd'] = candidates[driver_fc_cols].std(axis=1)

def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)

candidates['norm_abs_fc_all']    = minmax(candidates['log2FC'].abs())
candidates['norm_abs_fc_stage1'] = minmax(candidates['log2FC_stage1'].abs())
candidates['norm_consistency']   = 1 - minmax(candidates['cross_driver_sd'])
candidates['score'] = (0.40 * candidates['norm_abs_fc_all']
                     + 0.40 * candidates['norm_abs_fc_stage1']
                     + 0.20 * candidates['norm_consistency'])

top10 = candidates.sort_values('score', ascending=False).head(10)
top10.to_csv(OUT_PREFIX + 'KIRC_top10_candidates_LOCKED.csv', index=False)

# ── LOCK ────────────────────────────────────────────────────────────────────
LOCKED_CANDIDATES = top10['Ensembl_ID'].tolist()
LOCKED_SYMBOLS    = top10['gene_symbol'].tolist()

print("=== TOP 10 LOCKED CANDIDATES ===")
print(top10[['gene_symbol','log2FC','log2FC_stage1','score']].to_string(index=False))
print(f"\nLocked: {LOCKED_SYMBOLS}")


## 11. Expression Heatmap (Top 50 Core Genes)

In [ ]:
hm_top     = candidates.assign(abs_fc=candidates['log2FC'].abs()).nlargest(50, 'abs_fc')
hm_genes   = hm_top['Ensembl_ID'].tolist()
hm_symbols = hm_top['gene_symbol'].tolist()

hm_normal = fpkm_mito.set_index('Ensembl_ID').loc[hm_genes, normal_samples]
hm_tumor  = fpkm_mito.set_index('Ensembl_ID').loc[hm_genes, stage1_indep]
mu_n = hm_normal.mean(axis=1).values[:, None]
sd_n = hm_normal.std(axis=1).values[:, None] + 1e-9
hm_full = np.hstack([(hm_normal.values - mu_n)/sd_n, (hm_tumor.values - mu_n)/sd_n])

fig, ax = plt.subplots(figsize=(13, 12))
sns.heatmap(hm_full, cmap='RdBu_r', center=0, vmin=-4, vmax=4,
            yticklabels=hm_symbols, xticklabels=False,
            cbar_kws={'label': 'z-score vs normal mean/SD'}, ax=ax)
ax.axvline(len(normal_samples), color='black', lw=1.5)
ax.set_title(f"KIRC — top 50 core signature genes\n"
             f"Left: {len(normal_samples)} normals  |  Right: {len(stage1_indep)} Stage I tumours")
plt.tight_layout()
plt.savefig(OUT_PREFIX + 'KIRC_heatmap_z_to_normal.png', dpi=300, bbox_inches='tight')
plt.show()


## 12. Clinical Associations

In [ ]:
clin_results = []
for ens, sym in zip(LOCKED_CANDIDATES, LOCKED_SYMBOLS):
    gene_row = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples]
    df = pd.DataFrame({'sample': gene_row.index, 'expr': gene_row.values})
    df = df.merge(master[['sample','ajcc_pathologic_stage.diagnoses',
                           'age_at_diagnosis.diagnoses']], on='sample', how='left')
    groups = [g['expr'].values for _, g in
              df.dropna(subset=['ajcc_pathologic_stage.diagnoses']).groupby(
              'ajcc_pathologic_stage.diagnoses') if len(g) >= 5]
    p_stage = stats.kruskal(*groups).pvalue if len(groups) >= 2 else np.nan
    age_df  = df.dropna(subset=['age_at_diagnosis.diagnoses'])
    age_df['age'] = pd.to_numeric(age_df['age_at_diagnosis.diagnoses'], errors='coerce')
    age_df  = age_df.dropna(subset=['age'])
    rho, p_age = stats.spearmanr(age_df['expr'], age_df['age']) if len(age_df) >= 10 else (np.nan, np.nan)
    clin_results.append({'gene': sym, 'p_stage': p_stage, 'p_age': p_age, 'spearman_age': rho})

clin_df = pd.DataFrame(clin_results)
all_p   = clin_df[['p_stage','p_age']].values.flatten()
mask    = ~np.isnan(all_p)
adj     = np.full_like(all_p, np.nan, dtype=float)
adj[mask] = multipletests(all_p[mask], method='fdr_bh')[1]
clin_df['p_stage_adj'], clin_df['p_age_adj'] = adj.reshape(-1, 2).T
print(clin_df[['gene','p_stage_adj','p_age_adj']].to_string(index=False))
clin_df.to_csv(OUT_PREFIX + 'KIRC_candidate_clinical_associations.csv', index=False)
# Expected: 7/10 stage-significant; 0/10 age-significant


## 13. Survival Analysis — Kaplan-Meier and Cox Regression

In [ ]:
surv_df = survival[['sample','OS','OS.time']].copy().rename(
    columns={'OS': 'event', 'OS.time': 'time'})

km_results  = []
cox_results = []

for ens, sym in zip(LOCKED_CANDIDATES, LOCKED_SYMBOLS):
    expr_row = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples]
    df = (pd.DataFrame({'sample': expr_row.index, 'expr': expr_row.values})
            .merge(surv_df, on='sample', how='inner').dropna())
    if len(df) < 50: continue

    # Kaplan-Meier
    median_e = df['expr'].median()
    df['hi'] = (df['expr'] > median_e).astype(int)
    lr = logrank_test(df.loc[df['hi']==1,'time'], df.loc[df['hi']==0,'time'],
                      df.loc[df['hi']==1,'event'], df.loc[df['hi']==0,'event'])
    km_results.append({'gene': sym, 'logrank_p': lr.p_value})

    # Cox (adjusted for age + AJCC stage)
    cox_df_ = df.merge(master[['sample','ajcc_pathologic_stage.diagnoses',
                                'age_at_diagnosis.diagnoses']], on='sample', how='left')
    cox_df_['age']       = pd.to_numeric(cox_df_['age_at_diagnosis.diagnoses'], errors='coerce')
    cox_df_['stage_ord'] = cox_df_['ajcc_pathologic_stage.diagnoses'].map(
                           {'Stage I':1,'Stage II':2,'Stage III':3,'Stage IV':4})
    cox_df_ = cox_df_.dropna(subset=['expr','time','event','age','stage_ord'])
    try:
        cph = CoxPHFitter()
        cph.fit(cox_df_[['time','event','expr','age','stage_ord']],
                duration_col='time', event_col='event')
        cox_results.append({'gene': sym,
                            'HR_expr': cph.hazard_ratios_['expr'],
                            'p_cox':   cph.summary.loc['expr','p']})
    except:
        cox_results.append({'gene': sym, 'HR_expr': np.nan, 'p_cox': np.nan})

km_df  = pd.DataFrame(km_results)
cox_df = pd.DataFrame(cox_results)
km_df['logrank_p_adj'] = multipletests(km_df['logrank_p'],   method='fdr_bh')[1]
cox_df['p_cox_adj']    = multipletests(cox_df['p_cox'],      method='fdr_bh')[1]

print("KM results:")
print(km_df.to_string(index=False))
print("\nCox results:")
print(cox_df.to_string(index=False))
km_df.to_csv(OUT_PREFIX + 'KIRC_KM_results.csv', index=False)
cox_df.to_csv(OUT_PREFIX + 'KIRC_cox_results.csv', index=False)
# Expected: 7/10 KM significant; 5/10 Cox significant
# ALDH6A1: HR=0.674 (95% CI 0.561–0.810), adj p=1.3×10⁻⁴
# FKBP10:  HR=1.282, adj p=9.6×10⁻⁴


## 14. ROC Analysis

In [ ]:
# Individual candidate ROC (all 529 tumours vs 72 normals — Table 2 values)
roc_results = []
for ens, sym in zip(LOCKED_CANDIDATES, LOCKED_SYMBOLS):
    n_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, normal_samples].values
    t_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples].values
    y_true  = np.concatenate([np.zeros(len(n_e)), np.ones(len(t_e))])
    fpr, tpr, _ = roc_curve(y_true, np.concatenate([n_e, t_e]))
    a = auc(fpr, tpr)
    if a < 0.5: a = 1 - a
    roc_results.append({'gene': sym, 'AUC': round(a, 3)})

roc_df_tcga = pd.DataFrame(roc_results)
print(roc_df_tcga.to_string(index=False))
n_above_90 = (roc_df_tcga['AUC'] > 0.90).sum()
print(f"\nAUC > 0.90: {n_above_90}/10")  # expected: 9/10 (only FABP1 below)


## 15. Composite Suppression Score

In [ ]:
# Weights: normalised |log2FC| from TCGA all-stage (locked)
weights = top10.set_index('Ensembl_ID')['log2FC'].abs()
weights = weights / weights.sum()
weights_dict = weights.to_dict()
sym_weights  = {}
for ens, w in weights.items():
    row = candidates[candidates['Ensembl_ID'] == ens]
    if not row.empty:
        sym_weights[row.iloc[0]['gene_symbol']] = w

print("Locked composite weights:")
for ens, w in weights.items():
    sym = candidates.set_index('Ensembl_ID').loc[ens, 'gene_symbol']
    print(f"  {sym}: {w:.4f}")

cand_normal_expr = fpkm_mito.set_index('Ensembl_ID').loc[LOCKED_CANDIDATES, normal_samples]
mu_n = cand_normal_expr.mean(axis=1).values
sd_n = cand_normal_expr.std(axis=1).values

def composite_score(expr_mat, gene_list, wts, mu, sd):
    z = (expr_mat.values - mu[:, None]) / (sd[:, None] + 1e-9)
    w = np.array([wts[g] for g in gene_list])
    return (z * w[:, None]).sum(axis=0)

cand_tumor_expr  = fpkm_mito.set_index('Ensembl_ID').loc[LOCKED_CANDIDATES, tumor_samples]
tumor_scores  = composite_score(cand_tumor_expr,  LOCKED_CANDIDATES, weights_dict, mu_n, sd_n)
normal_scores = composite_score(cand_normal_expr, LOCKED_CANDIDATES, weights_dict, mu_n, sd_n)

y_true = np.concatenate([np.zeros(len(normal_scores)), np.ones(len(tumor_scores))])
fpr, tpr, _ = roc_curve(y_true, np.concatenate([normal_scores, tumor_scores]))
composite_auc = auc(fpr, tpr)
if composite_auc < 0.5: composite_auc = 1 - composite_auc
print(f"\nComposite score AUC: {composite_auc:.4f}")  # expected: 0.9676


## 16. GSEA Pathway Analysis

In [ ]:
import gseapy as gp

# Build MitoCarta pathway gene sets
mitocarta_pw = mitocarta[['Symbol','MitoCarta3.0_MitoPathways']].dropna()
pathway_sets = {}
for _, row in mitocarta_pw.iterrows():
    if isinstance(row['MitoCarta3.0_MitoPathways'], str):
        for p in row['MitoCarta3.0_MitoPathways'].split('|'):
            pathway_sets.setdefault(p.strip(), set()).add(row['Symbol'])
pathway_sets = {k: list(v) for k, v in pathway_sets.items() if 5 <= len(v) <= 500}
print(f"MitoCarta pathway gene sets: {len(pathway_sets)}")  # expected: 100

ranked = (de_all[['gene_symbol','rank_score']].dropna()
           .drop_duplicates('gene_symbol')
           .set_index('gene_symbol')['rank_score']
           .sort_values(ascending=False))

gsea = gp.prerank(rnk=ranked, gene_sets=pathway_sets,
                  permutation_num=1000, seed=RANDOM_SEED,
                  min_size=5, max_size=500, verbose=False)
gsea_res = gsea.res2d.sort_values('FDR q-val').head(15)
print(gsea_res[['Term','NES','NOM p-val','FDR q-val']].to_string(index=False))
gsea.res2d.to_csv(OUT_PREFIX + 'KIRC_GSEA_TCGA.csv', index=False)
# Expected: No pathway at FDR < 0.05 in TCGA-KIRC
# Top nominal: TCA cycle (FDR=0.071), BCAA (0.077), pyruvate (0.117), FAO (0.132)


## 17. External Validation

The complete MitoCarta DE pipeline is applied independently to each GEO cohort.
TCGA-KIRC results are **not referenced** during GEO analysis — full independence.

### Shared helper functions


In [ ]:
import GEOparse
from scipy.stats import hypergeom
from sklearn.metrics import roc_curve, auc as sk_auc

mito_syms    = set(mitocarta['Symbol'].dropna())
core_symbols = set(de_all.loc[de_all['Ensembl_ID'].isin(core_signature),
                               'gene_symbol'].dropna())
print(f"Core symbols for validation: {len(core_symbols)}")  # must be 70

def geo_full_de(expr_df, tumor_cols, normal_cols):
    rows = []
    for sym in expr_df.index:
        t = expr_df.loc[sym, tumor_cols].dropna().values
        n = expr_df.loc[sym, normal_cols].dropna().values
        if len(t) < 3 or len(n) < 3: continue
        _, p = stats.mannwhitneyu(n, t, alternative='two-sided')
        rows.append({'gene_symbol': sym,
                     'log2FC': np.mean(t) - np.mean(n), 'pvalue': p})
    df = pd.DataFrame(rows)
    _, padj, _, _ = multipletests(df['pvalue'], method='fdr_bh')
    df['pval_adj']    = padj
    df['sig_strict']  = (df['pval_adj'] < PADJ_THRESHOLD) & (df['log2FC'].abs() > LOG2FC_STRICT)
    df['sig_lenient'] = (df['pval_adj'] < PADJ_THRESHOLD) & (df['log2FC'].abs() > LOG2FC_LENIENT)
    df['rank_score']  = -np.log10(df['pval_adj'].replace(0, 1e-300)) * np.sign(df['log2FC'])
    return df

def geo_enrichment(geo_de, cohort, locked_syms, sym_wts,
                   expr_df, tumor_cols, normal_cols, pathway_sets):
    # Hypergeometric enrichment
    M     = len(geo_de)
    K     = len([s for s in core_symbols if s in geo_de['gene_symbol'].values])
    n_sig = geo_de['sig_strict'].sum()
    core_sig = geo_de.loc[geo_de['sig_strict'] &
                           geo_de['gene_symbol'].isin(core_symbols),
                           'gene_symbol'].tolist()
    k       = len(core_sig)
    p_hyper = hypergeom.sf(k-1, M, K, int(n_sig)) if n_sig > 0 else np.nan
    fe      = (k/int(n_sig))/(K/M) if n_sig > 0 and M > 0 else np.nan

    # Direction concordance
    concordant = sum(1 for s in core_sig
                     if not de_all.loc[de_all['gene_symbol']==s,'log2FC'].empty
                     and not geo_de.loc[geo_de['gene_symbol']==s,'log2FC'].empty
                     and np.sign(de_all.loc[de_all['gene_symbol']==s,'log2FC'].values[0])
                     == np.sign(geo_de.loc[geo_de['gene_symbol']==s,'log2FC'].values[0]))

    # Spearman ρ across core
    core_both   = [s for s in core_symbols if s in geo_de['gene_symbol'].values]
    tcga_fc_c   = [de_all.loc[de_all['gene_symbol']==s,'log2FC'].values[0] for s in core_both
                   if not de_all.loc[de_all['gene_symbol']==s,'log2FC'].empty]
    geo_fc_c    = [geo_de.loc[geo_de['gene_symbol']==s,'log2FC'].values[0] for s in core_both
                   if not geo_de.loc[geo_de['gene_symbol']==s,'log2FC'].empty]
    rho, p_rho  = stats.spearmanr(tcga_fc_c, geo_fc_c) if len(tcga_fc_c) > 5 else (np.nan, np.nan)

    # Composite AUC
    cand_here = [s for s in locked_syms if s in expr_df.index]
    n_e  = expr_df.loc[cand_here, normal_cols]
    mu_  = n_e.mean(axis=1).values; sd_ = n_e.std(axis=1).values + 1e-9
    w_   = np.array([sym_wts.get(s,0) for s in cand_here]); w_ /= (w_.sum()+1e-9)
    def sc(cols): return ((expr_df.loc[cand_here,cols].values - mu_[:,None]) / sd_[:,None] * w_[:,None]).sum(axis=0)
    t_sc = sc(tumor_cols); n_sc = sc(normal_cols)
    y_t  = np.concatenate([np.zeros(len(n_sc)), np.ones(len(t_sc))])
    fpr_, tpr_, _ = roc_curve(y_t, np.concatenate([n_sc, t_sc]))
    comp_auc = sk_auc(fpr_, tpr_)
    if comp_auc < 0.5: comp_auc = 1 - comp_auc
    _, p_mw  = stats.mannwhitneyu(n_sc, t_sc, alternative='two-sided')

    print(f"[{cohort}] k={k}/{K} (expected {K*int(n_sig)/M:.1f}), "
          f"p={p_hyper:.2e}, FE={fe:.1f}×, concordant={concordant}/{k}, "
          f"ρ={rho:.3f}, comp_AUC={comp_auc:.4f}")
    return {'hypergeom_p': p_hyper, 'FE': fe, 'concordant': f"{concordant}/{k}",
            'spearman_rho': rho, 'spearman_p': p_rho, 'comp_auc': comp_auc, 'comp_mw_p': p_mw}


### 17.1 GSE53757 (von Roemeling et al. 2013 — Affymetrix HG-U133 Plus 2.0)

In [ ]:
gse_53757 = GEOparse.get_GEO(geo='GSE53757', destdir=DATA_FOLDER, silent=True)
print(f"Loaded {gse_53757.name}: {len(gse_53757.gsms)} samples")

phenos53, expr_mat53 = [], {}
for gsm_name, gsm in gse_53757.gsms.items():
    title   = ' '.join(gsm.metadata.get('title', [])).lower()
    source  = ' '.join(gsm.metadata.get('source_name_ch1', [])).lower()
    chars   = ' '.join(gsm.metadata.get('characteristics_ch1', [])).lower()
    combined = f"{title} {source} {chars}"
    is_tumor = any(t in combined for t in ['tumor','tumour','ccrcc','carcinoma','cancer'])
    is_normal = any(t in combined for t in ['normal','non-tumor','non-tumour','adjacent'])
    if is_normal and not (is_tumor and 'tumor' in title): is_tumor = False
    stage = ''
    for c in gsm.metadata.get('characteristics_ch1', []):
        if 'tumor stage' in c.lower(): stage = c.split(':')[-1].strip()
    phenos53.append({'gsm': gsm_name, 'is_tumor': is_tumor, 'stage': stage})
    expr_mat53[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

pheno53  = pd.DataFrame(phenos53)
expr53   = pd.DataFrame(expr_mat53)
gpl53    = gse_53757.gpls['GPL570']
expr53['Symbol'] = expr53.index.map(gpl53.table.set_index('ID')['Gene Symbol'].to_dict())
expr53   = expr53.dropna(subset=['Symbol'])
expr53_g = expr53.groupby('Symbol').median()
print(f"Samples: {pheno53['is_tumor'].sum()}T / {(~pheno53['is_tumor']).sum()}N")
print(f"Genes after probe collapse: {expr53_g.shape[0]}")
# Log2-transform (MAS5 linear scale)
if expr53_g.max().max() > 30:
    expr53_g = np.log2(expr53_g.clip(lower=1))

t53 = pheno53.loc[pheno53['is_tumor'], 'gsm'].tolist()
n53 = pheno53.loc[~pheno53['is_tumor'], 'gsm'].tolist()
gse53757_mito = expr53_g[expr53_g.index.isin(mito_syms)].copy()

# PCA QC
X53 = StandardScaler().fit_transform(gse53757_mito[t53+n53].T.values)
pcs53 = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X53)
ev53  = PCA(n_components=2, random_state=RANDOM_SEED).fit(X53).explained_variance_ratio_*100
fig, ax = plt.subplots(figsize=(6,5))
for grp, col, mask_f in [('Normal','#2E86AB', lambda p: ~p['is_tumor']),
                          ('Tumour','#C73E1D', lambda p:  p['is_tumor'])]:
    m = np.array([i for i,r in pheno53.iterrows() if mask_f(pheno53).loc[i]])
    ax.scatter(pcs53[m,0], pcs53[m,1], c=col, alpha=0.6, s=18, label=grp)
ax.set_xlabel(f"PC1 — {ev53[0]:.1f}%"); ax.set_ylabel(f"PC2 — {ev53[1]:.1f}%")
ax.set_title("PCA — GSE53757"); ax.legend(); plt.tight_layout()
plt.savefig(OUT_PREFIX + 'KIRC_PCA_GSE53757.png', dpi=300); plt.show()

# Full DE
de_53757 = geo_full_de(gse53757_mito, t53, n53)
print(f"GSE53757 strict-sig: {de_53757['sig_strict'].sum()}")
de_53757.to_csv(OUT_PREFIX + 'KIRC_DE_GSE53757.csv', index=False)

# Enrichment
res53 = geo_enrichment(de_53757, 'GSE53757', LOCKED_SYMBOLS, sym_weights,
                       gse53757_mito, t53, n53, pathway_sets)
# Expected: k=65/77, p=1.56e-39, FE=4.5×, 65/65 concordant, ρ=0.785, AUC=0.9757


In [ ]:
# GSE53757 GSEA
ranked53 = (de_53757[['gene_symbol','rank_score']].dropna()
             .drop_duplicates('gene_symbol')
             .set_index('gene_symbol')['rank_score']
             .sort_values(ascending=False))
gsea53 = gp.prerank(rnk=ranked53, gene_sets=pathway_sets, permutation_num=1000,
                    seed=RANDOM_SEED, min_size=5, max_size=500, verbose=False)
print(gsea53.res2d.sort_values('FDR q-val').head(10)
           [['Term','NES','FDR q-val']].to_string(index=False))
gsea53.res2d.to_csv(OUT_PREFIX + 'KIRC_GSEA_GSE53757.csv', index=False)
# Expected: Trafficking NES=+2.19, FDR=0.021 (enriched)
# Carbohydrate, Itaconate, Glyoxylate, Fusion all depleted at FDR < 0.05


### 17.2 GSE40435 (Wozniak et al. 2013 — Illumina HumanHT-12 v4)

In [ ]:
gse_40435 = GEOparse.get_GEO(geo='GSE40435', destdir=DATA_FOLDER, silent=True)
print(f"Loaded {gse_40435.name}: {len(gse_40435.gsms)} samples")

phenos40, expr_mat40 = [], {}
for gsm_name, gsm in gse_40435.gsms.items():
    chars = ' '.join(gsm.metadata.get('characteristics_ch1', []))
    is_tumor = 'non-tumour' not in chars.lower() and 'adjacent' not in chars.lower()
    phenos40.append({'gsm': gsm_name, 'is_tumor': is_tumor, 'chars': chars})
    expr_mat40[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

pheno40 = pd.DataFrame(phenos40)
expr40  = pd.DataFrame(expr_mat40)
gpl40   = gse_40435.gpls['GPL10558']
expr40['Symbol'] = expr40.index.map(gpl40.table.set_index('ID')['Symbol'].to_dict())
expr40  = expr40.dropna(subset=['Symbol'])
expr40_g = expr40.groupby('Symbol').median()
print(f"Samples: {pheno40['is_tumor'].sum()}T / {(~pheno40['is_tumor']).sum()}N")
print(f"Genes after probe collapse: {expr40_g.shape[0]}")

t40 = pheno40.loc[pheno40['is_tumor'], 'gsm'].tolist()
n40 = pheno40.loc[~pheno40['is_tumor'], 'gsm'].tolist()
gse40435_mito = expr40_g[expr40_g.index.isin(mito_syms)].copy()

# PCA QC
X40  = StandardScaler().fit_transform(gse40435_mito[t40+n40].T.values)
pcs40 = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X40)
ev40  = PCA(n_components=2, random_state=RANDOM_SEED).fit(X40).explained_variance_ratio_*100
fig, ax = plt.subplots(figsize=(6,5))
for grp, col, mask_f in [('Normal','#2E86AB', lambda p: ~p['is_tumor']),
                          ('Tumour','#C73E1D', lambda p:  p['is_tumor'])]:
    m = np.array([i for i,r in pheno40.iterrows() if mask_f(pheno40).loc[i]])
    ax.scatter(pcs40[m,0], pcs40[m,1], c=col, alpha=0.6, s=18, label=grp)
ax.set_xlabel(f"PC1 — {ev40[0]:.1f}%"); ax.set_ylabel(f"PC2 — {ev40[1]:.1f}%")
ax.set_title("PCA — GSE40435"); ax.legend(); plt.tight_layout()
plt.savefig(OUT_PREFIX + 'KIRC_PCA_GSE40435.png', dpi=300); plt.show()

# Full DE
de_40435 = geo_full_de(gse40435_mito, t40, n40)
print(f"GSE40435 strict-sig: {de_40435['sig_strict'].sum()}")
de_40435.to_csv(OUT_PREFIX + 'KIRC_DE_GSE40435.csv', index=False)

# Grade associations
pheno40['age']   = pheno40['chars'].apply(
    lambda s: re.search(r'age:\s*(\d+)', s, re.I).group(1) if re.search(r'age:\s*(\d+)', s, re.I) else None)
pheno40['grade'] = pheno40['chars'].apply(
    lambda s: re.search(r'grade:\s*([IVX]+)', s, re.I).group(1) if re.search(r'grade:\s*([IVX]+)', s, re.I) else None)
pheno40['age'] = pd.to_numeric(pheno40['age'], errors='coerce')

clin40_rows = []
for sym in LOCKED_SYMBOLS:
    if sym not in gse40435_mito.index: continue
    df_c = pd.DataFrame({'gsm': t40, 'expr': gse40435_mito.loc[sym, t40].values})
    df_c = df_c.merge(pheno40[['gsm','age','grade']], on='gsm', how='left')
    grade_groups = [g['expr'].values for _, g in df_c.dropna(subset=['grade']).groupby('grade') if len(g) >= 5]
    p_grade = stats.kruskal(*grade_groups).pvalue if len(grade_groups) >= 2 else np.nan
    clin40_rows.append({'gene': sym, 'p_grade': p_grade})
clin40_df = pd.DataFrame(clin40_rows)
all_p40 = clin40_df['p_grade'].values
mask40  = ~np.isnan(all_p40)
adj40   = np.full_like(all_p40, np.nan, dtype=float)
adj40[mask40] = multipletests(all_p40[mask40], method='fdr_bh')[1]
clin40_df['p_grade_adj'] = adj40
print("\nGSE40435 grade associations:")
print(clin40_df.to_string(index=False))
clin40_df.to_csv(OUT_PREFIX + 'KIRC_clinical_GSE40435.csv', index=False)

# Enrichment
res40 = geo_enrichment(de_40435, 'GSE40435', LOCKED_SYMBOLS, sym_weights,
                       gse40435_mito, t40, n40, pathway_sets)
# Expected: k=44/75, p=1.54e-27, FE=5.5×, 44/44 concordant, ρ=0.618, AUC=0.9754


In [ ]:
# GSE40435 GSEA
ranked40 = (de_40435[['gene_symbol','rank_score']].dropna()
             .drop_duplicates('gene_symbol')
             .set_index('gene_symbol')['rank_score']
             .sort_values(ascending=False))
gsea40 = gp.prerank(rnk=ranked40, gene_sets=pathway_sets, permutation_num=1000,
                    seed=RANDOM_SEED, min_size=5, max_size=500, verbose=False)
print(gsea40.res2d.sort_values('FDR q-val').head(10)
           [['Term','NES','FDR q-val']].to_string(index=False))
gsea40.res2d.to_csv(OUT_PREFIX + 'KIRC_GSEA_GSE40435.csv', index=False)
# Expected: Trafficking NES=+2.09, FDR=0.022 (only pathway at FDR < 0.05)


## 18. DESeq2 Methodological Cross-Validation

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

counts = pd.read_csv(EXPR_COUNTS_FILE, sep='\t', compression='gzip')
counts['Ensembl_ID'] = counts['Ensembl_ID'].str.split('.').str[0]
counts = counts.drop_duplicates(subset='Ensembl_ID').set_index('Ensembl_ID')
counts_mito = counts.loc[counts.index.intersection(set(fpkm_mito['Ensembl_ID']))]
counts_mito = counts_mito.loc[(counts_mito > 0).any(axis=1)]

samples_c = [c for c in counts_mito.columns if c.endswith('-01A') or c.endswith('-11A')]
counts_sub = counts_mito[samples_c].T.astype(int)
metadata   = pd.DataFrame({
    'sample': samples_c,
    'condition': ['tumor' if s.endswith('-01A') else 'normal' for s in samples_c]
}).set_index('sample')

dds = DeseqDataSet(counts=counts_sub, metadata=metadata,
                   design_factors='condition', refit_cooks=True)
dds.deseq2()
res = DeseqStats(dds, contrast=['condition','tumor','normal'])
res.summary()
deseq_res = res.results_df.reset_index().rename(columns={'index':'Ensembl_ID'})

mw_  = de_all[['Ensembl_ID','log2FC','sig_strict']].rename(columns={'log2FC':'log2FC_MW'})
ds_  = deseq_res[['Ensembl_ID','log2FoldChange','padj']].rename(
       columns={'log2FoldChange':'log2FC_DESeq2'})
both = mw_.merge(ds_, on='Ensembl_ID', how='inner')
both['DESeq2_sig']      = both['padj'] < PADJ_THRESHOLD
both['concordant_sign'] = np.sign(both['log2FC_MW']) == np.sign(both['log2FC_DESeq2'])
print(f"Sign concordance: {100*both['concordant_sign'].mean():.1f}%")  # expected: 83.5%
cand_deseq = both[both['Ensembl_ID'].isin(LOCKED_CANDIDATES)].merge(
    gene_names, on='Ensembl_ID', how='left')[['gene_symbol','log2FC_MW','log2FC_DESeq2',
                                               'concordant_sign','DESeq2_sig']]
print("Candidate concordance:")
print(cand_deseq.to_string(index=False))
# Expected: 10/10 concordant, 10/10 significant in DESeq2


## 19. Pan-Cohort Convergence Analysis

In [ ]:
tcga_sig_syms = set(de_all.loc[de_all['sig_strict'], 'gene_symbol'].dropna())
geo53_sig_syms = set(de_53757.loc[de_53757['sig_strict'], 'gene_symbol'])
geo40_sig_syms = set(de_40435.loc[de_40435['sig_strict'], 'gene_symbol'])
pan_cohort = tcga_sig_syms & geo53_sig_syms & geo40_sig_syms
print(f"Pan-cohort genes (all 3 cohorts): {len(pan_cohort)}")  # expected: 51

# Direction concordance
discordant = 0
for sym in sorted(pan_cohort):
    t_fc = de_all.loc[de_all['gene_symbol']==sym,'log2FC']
    g53  = de_53757.loc[de_53757['gene_symbol']==sym,'log2FC']
    g40  = de_40435.loc[de_40435['gene_symbol']==sym,'log2FC']
    if not (t_fc.empty or g53.empty or g40.empty):
        signs = {np.sign(t_fc.values[0]), np.sign(g53.values[0]), np.sign(g40.values[0])}
        if len(signs) > 1: discordant += 1
print(f"Discordant: {discordant}")  # expected: 0

cand_pan = [s for s in LOCKED_SYMBOLS if s in pan_cohort]
print(f"Candidates in pan-cohort: {len(cand_pan)}/10 → {cand_pan}")
# Expected: 6/10 (HMGCS2, EFHD1, OGDHL, ALDH6A1, LDHD, ACSF2)

pan_df = de_all[de_all['gene_symbol'].isin(pan_cohort)][
    ['gene_symbol','Ensembl_ID','log2FC','pval_adj']].rename(
    columns={'log2FC':'log2FC_TCGA','pval_adj':'padj_TCGA'})
pan_df = (pan_df.merge(
    de_53757[['gene_symbol','log2FC','pval_adj']].rename(
        columns={'log2FC':'log2FC_GSE53757','pval_adj':'padj_GSE53757'}),
    on='gene_symbol', how='left').merge(
    de_40435[['gene_symbol','log2FC','pval_adj']].rename(
        columns={'log2FC':'log2FC_GSE40435','pval_adj':'padj_GSE40435'}),
    on='gene_symbol', how='left'))
pan_df.to_csv(OUT_PREFIX + 'KIRC_pan_cohort_51genes.csv', index=False)
print(f"Saved: {OUT_PREFIX}KIRC_pan_cohort_51genes.csv")


## 20. Supplementary Figures

In [ ]:
from matplotlib import gridspec

# ── Supplementary Figure S2: KM curves for all 7 significant candidates ──────
km_sig_genes = ['HMGCS2', 'FABP1', 'EFHD1', 'OGDHL', 'ALDH6A1', 'FKBP10', 'LDHD']

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.38)
for idx, sym in enumerate(km_sig_genes):
    row_, col_ = divmod(idx, 4)
    ax = fig.add_subplot(gs[row_, col_])
    ens = LOCKED_CANDIDATES[LOCKED_SYMBOLS.index(sym)]
    expr_row = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples]
    df_km = (pd.DataFrame({'sample': expr_row.index, 'expr': expr_row.values})
               .merge(surv_df, on='sample', how='inner').dropna())
    df_km['hi'] = (df_km['expr'] > df_km['expr'].median()).astype(int)
    for grp, c_hex, ls, lbl in [(1,'#2E86AB','-','High'),(0,'#C73E1D','--','Low')]:
        m = df_km['hi'] == grp
        KaplanMeierFitter().fit(
            df_km.loc[m,'time'], df_km.loc[m,'event'],
            label=f'{lbl} (n={m.sum()})').plot_survival_function(
            ax=ax, ci_show=True, color=c_hex, linestyle=ls)
    adj_p = km_df.loc[km_df['gene']==sym,'logrank_p_adj'].values[0]
    ax.set_title(f'{sym}\nadj p = {adj_p:.2e}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (days)', fontsize=8); ax.set_ylabel('Survival probability', fontsize=8)
    ax.set_ylim(0, 1.05); ax.legend(fontsize=7, loc='lower left')
    ax.tick_params(labelsize=7); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax8 = fig.add_subplot(gs[1, 3]); ax8.axis('off')
ax8.text(0.5, 0.5, 'TCGA-KIRC\nMedian split\nBH-FDR corrected\nacross 10 tests',
         ha='center', va='center', fontsize=9, style='italic', color='gray',
         transform=ax8.transAxes)
fig.suptitle('Supplementary Figure S2 | Kaplan-Meier Overall Survival — '
             'All 7 Significant Candidates (TCGA-KIRC)', fontsize=11, fontweight='bold', y=1.01)
plt.savefig(OUT_PREFIX + 'SuppFig_S2_KM_all7.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: SuppFig_S2_KM_all7.png")


In [ ]:
# ── Supplementary Figure S3: Individual ROC curves for all 10 candidates ─────
fig = plt.figure(figsize=(20, 9))
gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.42, wspace=0.38)
for idx, (ens, sym) in enumerate(zip(LOCKED_CANDIDATES, LOCKED_SYMBOLS)):
    row_, col_ = divmod(idx, 5)
    ax = fig.add_subplot(gs[row_, col_])
    n_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, normal_samples].values
    t_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples].values
    y_t = np.concatenate([np.zeros(len(n_e)), np.ones(len(t_e))])
    fpr_, tpr_, _ = roc_curve(y_t, np.concatenate([n_e, t_e]))
    a = auc(fpr_, tpr_)
    if a < 0.5: a = 1 - a; fpr_ = 1-fpr_[::-1]; tpr_ = 1-tpr_[::-1]
    ax.plot(fpr_, tpr_, color='#2E86AB', lw=1.8, label=f'AUC = {a:.3f}')
    ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5)
    ax.fill_between(fpr_, tpr_, alpha=0.10, color='#2E86AB')
    ax.set_title(sym, fontsize=11, fontweight='bold')
    ax.set_xlabel('1 − Specificity', fontsize=8); ax.set_ylabel('Sensitivity', fontsize=8)
    ax.legend(fontsize=8, loc='lower right'); ax.tick_params(labelsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
fig.suptitle('Supplementary Figure S3 | Individual ROC Curves — All 10 Candidate Genes '
             '(TCGA-KIRC, 529 Tumours vs 72 Normals)', fontsize=11, fontweight='bold', y=1.01)
plt.savefig(OUT_PREFIX + 'SuppFig_S3_ROC_all10.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: SuppFig_S3_ROC_all10.png")


## 21. Manuscript Verification Summary

Reproduces all key statistics reported in the manuscript and flags any discrepancies.
Run this cell as a final check before submission.


In [ ]:
print("=" * 65)
print("MANUSCRIPT VERIFICATION SUMMARY")
print("=" * 65)

checks = []
def check(label, condition, note=""):
    symbol = "✓" if condition else "✗ FAIL"
    print(f"  {symbol}  {label}" + (f"  [{note}]" if note else ""))
    checks.append(condition)

print("\n── Cohort sizes ──")
check("Tumours = 529",        len(tumor_samples) == 529)
check("Normals = 72",         len(normal_samples) == 72)
check("Stage I indep = 246",  len(stage1_indep) == 246)
check("Stage I total = 270",  len(stage1_in_expr) == 270)

print("\n── DE results ──")
check("All-stage strict = 111",  de_all['sig_strict'].sum() == 111)
check("93 downregulated",        ((de_all['sig_strict'])&(de_all['log2FC']<0)).sum() == 93)
check("18 upregulated",          ((de_all['sig_strict'])&(de_all['log2FC']>0)).sum() == 18)
check("Lenient = 369",           de_all['sig_lenient'].sum() == 369)
check("Stage I strict = 101",    de_stage1_indep['sig_strict'].sum() == 101)
check("Stage I/all-stage overlap = 98",
      len(set(de_stage1_indep.loc[de_stage1_indep['sig_strict'],'Ensembl_ID']) &
          set(de_all.loc[de_all['sig_strict'],'Ensembl_ID'])) == 98)

print("\n── Driver independence ──")
check("VHL n_mut = 84",   driver_results['VHL']['n_mut'] == 84)
check("VHL n_intact = 95", driver_results['VHL']['n_intact'] == 95)
check("VHL-indep = 81",   len(driver_results['VHL']['independent']) == 81)
check("PBRM1 n_mut = 71", driver_results['PBRM1']['n_mut'] == 71)
check("PBRM1-indep = 75", len(driver_results['PBRM1']['independent']) == 75)
check("Core = 70",        len(core_signature) == 70)
check("Sensitivity = 62", len(core_signature_setd2) == 62)

print("\n── Table 2 (all-tumour ROC) ──")
expected_auc = {'HMGCS2':0.919,'FABP1':0.809,'NAT8L':0.975,'EFHD1':0.980,
                'OGDHL':0.913,'COX4I2':0.922,'ALDH6A1':0.937,'FKBP10':0.943,
                'LDHD':0.917,'ACSF2':0.901}
for ens, sym in zip(LOCKED_CANDIDATES, LOCKED_SYMBOLS):
    n_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, normal_samples].values
    t_e = fpkm_mito.set_index('Ensembl_ID').loc[ens, tumor_samples].values
    fpr_, tpr_, _ = roc_curve(np.concatenate([np.zeros(len(n_e)), np.ones(len(t_e))]),
                               np.concatenate([n_e, t_e]))
    a = auc(fpr_, tpr_)
    if a < 0.5: a = 1 - a
    check(f"{sym} AUC = {expected_auc[sym]}", abs(a - expected_auc[sym]) < 0.005)

check("9/10 AUC > 0.90",
      sum(1 for ens,sym in zip(LOCKED_CANDIDATES,LOCKED_SYMBOLS)
          for a_ in [auc(*roc_curve(np.concatenate([np.zeros(72),np.ones(529)]),
              np.concatenate([fpkm_mito.set_index('Ensembl_ID').loc[ens,normal_samples].values,
                              fpkm_mito.set_index('Ensembl_ID').loc[ens,tumor_samples].values]))[0:2])]
          if (1-a_ if a_<0.5 else a_) > 0.90) == 9)

print(f"\n── Composite AUC ──")
check(f"Composite AUC = 0.9676", abs(composite_auc - 0.9676) < 0.0005)

print("\n── Survival ──")
km_sig_n  = sum(v < 0.05 for v in {r['gene']:r['logrank_p_adj'] for _,r in km_df.iterrows()}.values())
cox_sig_n = sum(v < 0.05 for v in {r['gene']:r['p_cox_adj']    for _,r in cox_df.iterrows()}.values())
check("7/10 KM significant",  km_sig_n  == 7)
check("5/10 Cox significant", cox_sig_n == 5)
check("ALDH6A1 HR ≈ 0.674",
      abs(cox_df.loc[cox_df['gene']=='ALDH6A1','HR_expr'].values[0] - 0.674) < 0.001)
check("FKBP10 HR ≈ 1.282",
      abs(cox_df.loc[cox_df['gene']=='FKBP10','HR_expr'].values[0] - 1.282) < 0.001)

print("\n── Validation ──")
check("DESeq2 sign concordance = 83.5%", abs(both['concordant_sign'].mean()*100 - 83.5) < 0.1)
check("Pan-cohort genes = 51", len(pan_cohort) == 51)
check("Pan-cohort discordant = 0", discordant == 0)

print("\n" + "=" * 65)
passed = sum(checks)
total  = len(checks)
print(f"RESULT: {passed}/{total} checks passed" +
      ("  ✓  ALL CORRECT" if passed == total else f"  ✗  {total-passed} FAILED"))
print("=" * 65)
